<a href="https://colab.research.google.com/github/Kelvin-Wepo/Rafiki.ai/blob/main/SadTalker_GPU_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Complete Colab Setup (ONE CELL - Run in Google Colab ONLY)
import os
import subprocess
import sys

# Verify we're in Colab
try:
    from google.colab import drive
    print("✅ Running in Google Colab")
except:
    print("❌ ERROR: This must run in Google Colab, not VS Code!")
    print("Go to: https://colab.research.google.com")
    raise SystemExit("Not running in Colab")

# Check GPU
import torch
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → T4 GPU")
    raise SystemExit("No GPU detected")

print("\n📦 Installing dependencies...")
# Install older compatible versions
!pip install -q torch==1.12.1+cu116 torchvision==0.13.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116
!pip install -q gradio==3.50.2 imageio-ffmpeg safetensors pyngrok
!pip install -q kornia==0.6.8 yacs face-alignment imageio scikit-image
!pip install -q basicsr realesrgan gfpgan

# Clone SadTalker
if not os.path.exists('/content/SadTalker'):
    print("📥 Cloning SadTalker...")
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker

os.chdir('/content/SadTalker')

# Download models
if not os.path.exists('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    print("📥 Downloading models...")
    !bash scripts/download_models.sh

!pip install -q -r requirements.txt

# Create compatibility patches
patch = """
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
for attr in ['VisibleDeprecationWarning','ModuleDeprecationWarning','ComplexWarning']:
    if not hasattr(np, attr): setattr(np, attr, UserWarning)
for attr in ['bool','int','float','complex','object','str']:
    if not hasattr(np, attr): setattr(np, attr, eval(attr))

import torchvision.transforms.functional as TF
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    import types
    ft = types.ModuleType('functional_tensor')
    for a in dir(TF):
        if not a.startswith('_'): setattr(ft, a, getattr(TF, a))
    sys.modules['torchvision.transforms.functional_tensor'] = ft
    import torchvision.transforms
    torchvision.transforms.functional_tensor = ft
"""

with open('/content/SadTalker/compat.py', 'w') as f:
    f.write(patch)

# Create server
server = """#!/usr/bin/env python3
import os, sys
os.chdir('/content/SadTalker')
sys.path.insert(0, '/content/SadTalker')
import compat

from src.gradio_demo import SadTalker
from pyngrok import ngrok
import gradio as gr
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
ngrok.set_auth_token("2bbP4fASCJKmNwWFSSbyQybSMxT_vQ4HRYiotjCLzY1LHByR")

sad_talker = SadTalker(checkpoint_path='checkpoints', config_path='src/config', lazy_load=True)

def gen(img, aud, pre='crop', still=False, exp=1.0, sz=256, enh=False):
    return sad_talker.test(source_image=img, driven_audio=aud, preprocess=pre,
                          still_mode=still, expression_scale=exp,
                          enhancer='gfpgan' if enh else None, batch_size=2, size=sz, pose_style=0)

ui = gr.Interface(fn=gen,
    inputs=[gr.Image(type="filepath"), gr.Audio(type="filepath"),
            gr.Dropdown(['crop','resize','full'], value='crop'),
            gr.Checkbox(False), gr.Slider(0,2,1,0.1),
            gr.Dropdown([256,512], value=256), gr.Checkbox(False)],
    outputs=gr.Video(), title="SadTalker GPU Server")

url = ngrok.connect(7860)
print(f"\\n{'='*70}\\n🚀 SERVER RUNNING!\\n{'='*70}\\n📡 URL: {url}\\n{'='*70}\\n")
ui.launch(server_port=7860, share=False)
"""

with open('/content/server.py', 'w') as f:
    f.write(server)

print("\n🚀 Starting server...\n")
!python3 /content/server.py

✅ Running in Google Colab
✅ CUDA available: True
✅ GPU: Tesla T4

📦 Installing dependencies...
ERROR: Could not find a version that satisfies the requirement torch==1.12.1+cu116 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==1.12.1+cu116
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires num

## Important: Restart Runtime After Setup

After running Cell 1, you may need to restart the runtime to apply numpy fixes.

**Go to: Runtime → Restart runtime**

Then continue with Cell 2 and Cell 3.

In [1]:
# Cell 2: Configure ngrok token

# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "2bbP4fASCJKmNwWFSSbyQybSMxT_vQ4HRYiotjCLzY1LHByR"  # ⚠️ REPLACE THIS!

if NGROK_TOKEN == "2bbP4fASCJKmNwWFSSbyQybSMxT_vQ4HRYiotjCLzY1LHByR":
    print("⚠️  Please set your ngrok token above!")
    print("Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok token configured!")

⚠️  Please set your ngrok token above!
Get it from: https://dashboard.ngrok.com/get-started/your-authtoken


In [ ]:
import sys
import os

sadtalker_dir = '/content/SadTalker'
if not os.path.exists(sadtalker_dir):
    print(f"ERROR: SadTalker directory not found at {sadtalker_dir}.")
    print("Please ensure Cell 1 (Setup and Installation) has been run successfully and the SadTalker repository is cloned.")
    raise FileNotFoundError(f"SadTalker directory '{sadtalker_dir}' not found. Please run Cell 1.")
else:
    os.chdir(sadtalker_dir)
sys.path.insert(0, sadtalker_dir)

from src.gradio_demo import SadTalker
from pyngrok import ngrok
import gradio as gr
import torch

print("🚀 Initializing SadTalker with GPU...")

# Initialize SadTalker
sad_talker = SadTalker(
    checkpoint_path='checkpoints',
    config_path='src/config',
    lazy_load=True
)

print(f"✅ SadTalker initialized on {torch.cuda.get_device_name(0)}")

def generate_video(source_image, driven_audio, preprocess='crop',
                  still_mode=False, expression_scale=1.0, size=256, enhancer=False):
    """Generate talking head video with GPU acceleration"""
    try:
        print(f"\n🎬 Generating video...")
        print(f"   Size: {size}x{size}")
        print(f"   Preprocess: {preprocess}")
        print(f"   Expression scale: {expression_scale}")

        result = sad_talker.test(
            source_image=source_image,
            driven_audio=driven_audio,
            preprocess=preprocess,
            still_mode=still_mode,
            expression_scale=expression_scale,
            enhancer='gfpgan' if enhancer else None,
            batch_size=2,
            size=size,
            pose_style=0
        )

        print(f"✅ Video generated: {result}")
        return result

    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        print(error_msg)
        return None

# Create Gradio interface
interface = gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Image(type="filepath", label="Avatar Image"),
        gr.Audio(type="filepath", label="Audio (WAV or MP3)"),
        gr.Dropdown(['crop', 'resize', 'full'], value='crop', label="Preprocessing"),
        gr.Checkbox(value=False, label="Still Mode (less head movement)"),
        gr.Slider(0.0, 2.0, value=1.0, step=0.1, label="Expression Scale"),
        gr.Dropdown([256, 512], value=256, label="Video Size"),
        gr.Checkbox(value=False, label="Face Enhancer (slower)")
    ],
    outputs=gr.Video(label="Generated Video"),
    title="🎬 SadTalker GPU Server - Rafiki AI",
    description="Fast lip-sync video generation powered by Google Colab GPU",
    examples=[
        ["examples/source_image/full_body_1.png", "examples/driven_audio/bus_chinese.wav", "crop", False, 1.0, 256, False]
    ]
)

# Start with ngrok tunnel
public_url = ngrok.connect(7860)

print("\n" + "="*70)
print("🚀 SadTalker GPU API Server is RUNNING!")
print("="*70)
print(f"\n📡 Public URL: {public_url}")
print(f"\n⚠️  IMPORTANT: Copy this URL to your backend configuration!")
print(f"\nSet this in your backend:")
print(f"  export SADTALKER_API_URL='{public_url}'")
print(f"  export SADTALKER_MODE='api'")
print("\n" + "="*70)

# Launch Gradio
interface.launch(share=False, server_port=7860)

ERROR: SadTalker directory not found at /content/SadTalker.
Please ensure Cell 1 (Setup and Installation) has been run successfully and the SadTalker repository is cloned.


FileNotFoundError: SadTalker directory '/content/SadTalker' not found. Please run Cell 1.

## Usage Instructions

Once the server is running:

1. **Copy the public URL** from the output above
2. **Update your backend** (`/home/subchief/5TECH/backend/config.py`):
   ```python
   SADTALKER_API_URL = "https://xxxx.ngrok.io"  # Your ngrok URL
   SADTALKER_MODE = "api"  # Use API mode
   ```
3. **Restart your backend** to use the GPU server

### Performance:
- **With GPU (T4):** 5-10 seconds per video
- **CPU only:** 2-10 minutes per video
- **50-100x speedup!** 🚀

### Notes:
- Keep this Colab tab open while using the API
- Free Colab sessions last up to 12 hours
- The ngrok URL changes each time you restart